# Logistic Regression

Logistic Regression is a **binary classifier** that models the probability of class 1 using the **sigmoid** function and minimises **binary cross-entropy** via gradient descent.

**Dataset:** Breast Cancer Wisconsin — Malignant vs Benign.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, '.')
from logistic_regression import LogisticRegression
np.random.seed(42)
print("Imports complete")

## Load & Explore the Data

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Classes: {list(class_names)}  (0=Malignant, 1=Benign)")
print(f"Distribution: Malignant={( y==0).sum()}, Benign={(y==1).sum()}")

## Preprocess & Split

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## Fit the Model

In [ ]:
model = LogisticRegression(learning_rate=0.1, n_iterations=1000)
model.fit(X_train, y_train)
print(f"\nWeight vector shape: {model.weights.shape}")
print(f"Bias: {model.bias:.4f}")

## Training Loss Curve

In [ ]:
# Re-compute loss history
epsilon = 1e-15
losses = []
w, b = np.zeros(X_train.shape[1]), 0.0
lr = 0.1
for i in range(1000):
    z = X_train @ w + b
    p = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    p_clip = np.clip(p, epsilon, 1-epsilon)
    loss = -np.mean(y_train * np.log(p_clip) + (1-y_train) * np.log(1-p_clip))
    losses.append(loss)
    err = p - y_train
    w -= lr * (X_train.T @ err) / len(y_train)
    b -= lr * err.mean()

plt.figure(figsize=(8,4))
plt.plot(losses, color='steelblue', linewidth=1.5)
plt.xlabel("Iteration"); plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Training Loss Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Initial loss: {losses[0]:.4f} | Final loss: {losses[-1]:.4f}")

## Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

train_acc = model.score(X_train, y_train)
test_acc  = model.score(X_test,  y_test)
y_pred    = model.predict(X_test)
y_proba   = model.predict_proba(X_test)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test  Accuracy: {test_acc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=class_names))

## Confusion Matrix & ROC Curve

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13,5))

# Confusion matrix
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(class_names); axes[0].set_yticklabels(class_names)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i,j], ha='center', va='center', fontsize=14,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')

# ROC
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC={roc_auc:.4f}')
axes[1].plot([0,1],[0,1], 'k--', linewidth=1)
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Top Feature Weights

In [ ]:
weights = np.abs(model.weights)
top_idx = np.argsort(weights)[-10:]

plt.figure(figsize=(9,5))
plt.barh(feature_names[top_idx], model.weights[top_idx],
         color=['tomato' if w < 0 else 'steelblue' for w in model.weights[top_idx]],
         edgecolor='white')
plt.axvline(0, color='gray', linewidth=0.8)
plt.xlabel("Weight")
plt.title("Top 10 Logistic Regression Weights")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

- Logistic Regression applies a **sigmoid** to a linear model to produce probabilities.
- Binary cross-entropy penalises confident wrong predictions more heavily.
- Positive weights push toward class 1 (Benign); negative toward class 0 (Malignant).
- AUC near 1.0 indicates excellent separation between classes.
